# V2: 실제 셀 분리 학습·보정·팩 구성 비교
변경 기록은 `CHANGES_V2.txt`에 있습니다. 32셀 학습 + 11셀 구간 보정 + 43셀 평가로 분리합니다.
서로 다른 시드 3개와 외부 2-fold의 결과이며, 이전 86셀 풀 점수와 직접 비교하지 않습니다.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
assert (ROOT / 'configs/model_comparison_v2.json').is_file()
config = json.loads((ROOT / 'configs/model_comparison_v2.json').read_text())
OUT = ROOT / 'results' / config['run_name']
config

In [ ]:
display(pd.read_csv(OUT / 'comparison_deltas.csv').round(2))
display(Image(filename=str(OUT / 'v2_model_comparison.png')))
display(Image(filename=str(OUT / 'v2_calibration.png')))

## 결과 해석
학습 모델은 RUL 오차를 줄였지만 팩 점수 개선은 관측 조건에 따라 달랐습니다.
81사이클에서는 기존 선형모델보다 낮았습니다. 셀 교환도 일관된 실제 이득을 주지 못했습니다.
구간 보정은 포함률을 높였지만 구간 폭도 늘었습니다. 예측 정확도, 분포의 신뢰도와 의사결정 성능을 따로 확인합니다.
이 버전은 예측·팩 구성 비교이며 새로운 검사 선택 또는 중단 정책의 성과가 아닙니다.

In [ ]:
display(pd.read_csv(OUT / 'summary.csv').round(3))
display(json.loads((OUT / 'audit.json').read_text()))
display(json.loads((OUT / 'splits.json').read_text())[0])

## 다시 실행
설정을 바꿀 때는 `run_name`을 바꾸면 기존 결과를 보존할 수 있습니다. 아래 셀은 비교와 결과 재계산 검증을 실행합니다.

In [ ]:
subprocess.run([sys.executable, '-m', 'battery_sim.compare_models'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'battery_sim.audit_comparison'], cwd=ROOT, check=True)